# Import the library

In [1]:
import os

import torch
from trainer import Trainer, TrainerArgs

# from TTS.bin.compute_embeddings import compute_embeddings
from compute_embeddings import compute_embeddings # use custom formatter without forking the lib
from TTS.bin.resample import resample_files
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import CharactersConfig, Vits, VitsArgs, VitsAudioConfig
# from TTS.utils.downloaders import download_vctk
# from TTS.config import load_config
# from TTS.config.shared_configs import BaseDatasetConfig
# from TTS.tts.datasets import load_tts_samples
# from TTS.tts.utils.managers import save_file
# from TTS.tts.utils.speakers import SpeakerManager
from TTS.tts.datasets.formatters import vctk
from functools import partial

from tqdm import tqdm

torch.set_num_threads(24)

# Setup constants

In [2]:
# Current path
CURRENT_PATH = os.getcwd()

# Name of the run for the Trainer
RUN_NAME = "KhongKhunTTS-Experiment04-CV_and_TC"

# Path where you want to save the models outputs (configs, checkpoints and tensorboard logs)
OUT_PATH = os.path.join(CURRENT_PATH, "runs")

# If you want to do transfer learning and speedup your training you can set here the path to the model
RESTORE_PATH = "./best_model_base.pth"

# This paramter is useful to debug, it skips the training epochs and just do the evaluation  and produce the test sentences
SKIP_TRAIN_EPOCH = False

# Set here the batch size to be used in training and evaluation
BATCH_SIZE = 32

# Training Sampling rate and the target sampling rate for resampling the downloaded dataset (Note: If you change this you might need to redownload the dataset !!)
# Note: If you add new datasets, please make sure that the dataset sampling rate and this parameter are matching, otherwise resample your audios
SAMPLE_RATE = 16000

# Max audio length in seconds to be used in training (every audio bigger than it will be ignored)
MAX_AUDIO_LEN_IN_SECONDS = 10

# Define the number of threads used during the audio resampling
NUM_RESAMPLE_THREADS = 10

# Dataset configuration

In [ ]:
# init configs
commonvoice_config = BaseDatasetConfig(
    formatter="vctk",
    dataset_name="commonvoice",
    meta_file_train="",
    meta_file_val="",
    path=os.path.join(CURRENT_PATH, "commonvoice-to-vctk"),
    language="th",
    ignored_speakers=[
        "cv017", # Female Teenager
        "cv048", # Female Teenager
        "cv039", # Female Adult
        "cv052", # Female Adult
        "cv069", # Male Teenager
        "cv054", # Male Teenager
        "cv049", # Male Adult
        "cv026", # Male Adult
    ], # For testing set
)

thaicentral_config = BaseDatasetConfig(
    formatter="vctk",
    dataset_name="thaicentral",
    meta_file_train="",
    meta_file_val="",
    path=os.path.join(CURRENT_PATH, "thai-central-to-vctk"),
    language="th",
    ignored_speakers=[
        "tc0149", # Female Teenager
        "tc0020", # Female Teenager
        "tc0018", # Female Adult
        "tc0080", # Female Adult
        "tc0074", # Male Teenager
        "tc0152", # Male Teenager
        "tc0200", # Male Adult
        "tc0099", # Male Adult

        "tc0622", # Invalid
        "tc0564", # Invalid
        "tc0812", # Invalid
        "tc0441", # Invalid
        "tc0508", # Invalid
        "tc0698", # Invalid
        "tc0412", # Invalid
        "tc0611", # Invalid
        "tc0659", # Invalid
    ], # For testing set
)

# Add here all datasets configs, in our case we just want to train with the VCTK dataset then we need to add just VCTK. Note: If you want to add new datasets, just add them here and it will automatically compute the speaker embeddings (d-vectors) for this new dataset :)
DATASETS_CONFIG_LIST = [commonvoice_config, thaicentral_config]

# Setup custom formatter

In [4]:
from TTS.tts.datasets.formatters import vctk
import TTS.tts.datasets.formatters as formatters_module 

# Save original function
original_vctk = vctk

def vctk_16k(root_path, meta_files=None, ignored_speakers=None):
    return original_vctk(
        root_path, 
        meta_files=meta_files, 
        wavs_path="wav16_silence_trimmed",
        mic="mic1",
        ignored_speakers=ignored_speakers
    )

# Replace in multiple places to be sure
import sys

# 1. Replace in formatters module
setattr(formatters_module, 'vctk', vctk_16k)

# 2. Replace in sys.modules
tts_formatters = sys.modules['TTS.tts.datasets.formatters']
setattr(tts_formatters, 'vctk', vctk_16k)

# 3. Also replace in the parent module
tts_datasets = sys.modules['TTS.tts.datasets']
if hasattr(tts_datasets, 'vctk'):
    setattr(tts_datasets, 'vctk', vctk_16k)

# 4. Replace in the current module's namespace
import TTS.tts.datasets as datasets
if hasattr(datasets, 'vctk'):
    setattr(datasets, 'vctk', vctk_16k)

# Extract speaker embeddings

In [5]:
SPEAKER_ENCODER_CHECKPOINT_PATH = (
    "https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/model_se.pth.tar"
)
SPEAKER_ENCODER_CONFIG_PATH = "https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/config_se.json"

D_VECTOR_FILES = []  # List of speaker embeddings/d-vectors to be used during the training

# Iterates all the dataset configs checking if the speakers embeddings are already computated, if not compute it
for dataset_conf in DATASETS_CONFIG_LIST:
    # Check if the embeddings weren't already computed, if not compute it
    embeddings_file = os.path.join(dataset_conf.path, "dvector.pth")
    if not os.path.isfile(embeddings_file):
        print(f">>> Computing the speaker embeddings for the {dataset_conf.dataset_name} dataset")
        compute_embeddings(
            SPEAKER_ENCODER_CHECKPOINT_PATH,
            SPEAKER_ENCODER_CONFIG_PATH,
            embeddings_file,
            formatter_name=dataset_conf.formatter,
            # formatter=vctk_16k if dataset_conf.formatter == "vctk_16k" else None,
            dataset_name=dataset_conf.dataset_name,
            dataset_path=dataset_conf.path,
            meta_file_train=dataset_conf.meta_file_train,
            meta_file_val=dataset_conf.meta_file_val,
        )

    D_VECTOR_FILES.append(embeddings_file)

# Audio config used in training.

In [6]:
audio_config = VitsAudioConfig(
    sample_rate=SAMPLE_RATE,
    hop_length=256,
    win_length=1024,
    fft_size=1024,
    mel_fmin=0.0,
    mel_fmax=None,
    num_mels=80,
)

# Model configuration

In [7]:
# Init VITSArgs setting the arguments that are needed for the KhongKhunTTS model
model_args = VitsArgs(
    d_vector_file=D_VECTOR_FILES,
    use_d_vector_file=True,
    d_vector_dim=512,
    num_layers_text_encoder=10,
    speaker_encoder_model_path=SPEAKER_ENCODER_CHECKPOINT_PATH,
    speaker_encoder_config_path=SPEAKER_ENCODER_CONFIG_PATH,
    resblock_type_decoder="2",  # In the YourTTS paper, trained using ResNet blocks type 2, if you like you can use the ResNet blocks type 1 like the VITS model
    # Useful parameters to enable the Speaker Consistency Loss (SCL) described in the paper
    use_speaker_encoder_as_loss=True,
    # Useful parameters to enable multilingual training
    use_language_embedding=True,
    embedded_language_dim=4,
)

In [8]:
# General training config, here you can change the batch size and others useful parameters
config = VitsConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name=RUN_NAME,
    project_name="KhongKhunTTS",
    run_description="""
            - KhongKhunTTS trained using ThaiCentral and CommonVoiceTH (VCTK structure) with transfer learning from TSync2
        """,
    dashboard_logger="tensorboard",
    logger_uri=None,
    audio=audio_config,
    batch_size=BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=BATCH_SIZE,
    num_loader_workers=8,
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=1000,
    save_step=5000,
    save_n_checkpoints=2,
    save_checkpoints=True,
    target_loss="loss_1",
    print_eval=False,
    use_phonemes=False,
    phonemizer="espeak",
    phoneme_language="en",
    compute_input_seq_cache=True,
    add_blank=True,
    text_cleaner="multilingual_cleaners",
    characters=CharactersConfig(
        characters_class="TTS.tts.models.vits.VitsCharacters",
        pad="_",
        eos="&",
        bos="*",
        blank=None,
        characters="ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz\u00af\u00b7\u00df\u00e0\u00e1\u00e2\u00e3\u00e4\u00e6\u00e7\u00e8\u00e9\u00ea\u00eb\u00ec\u00ed\u00ee\u00ef\u00f1\u00f2\u00f3\u00f4\u00f5\u00f6\u00f9\u00fa\u00fb\u00fc\u00ff\u0101\u0105\u0107\u0113\u0119\u011b\u012b\u0131\u0142\u0144\u014d\u0151\u0153\u015b\u016b\u0171\u017a\u017c\u01ce\u01d0\u01d2\u01d4\u0430\u0431\u0432\u0433\u0434\u0435\u0436\u0437\u0438\u0439\u043a\u043b\u043c\u043d\u043e\u043f\u0440\u0441\u0442\u0443\u0444\u0445\u0446\u0447\u0448\u0449\u044a\u044b\u044c\u044d\u044e\u044f\u0451\u0454\u0456\u0457\u0491\u2013!\"'(),-.:;?|~ \u0e01\u0e02\u0e04\u0e06\u0e07\u0e08\u0e09\u0e0a\u0e0b\u0e0c\u0e0d\u0e0e\u0e0f\u0e10\u0e11\u0e12\u0e13\u0e14\u0e15\u0e16\u0e17\u0e18\u0e19\u0e1a\u0e1b\u0e1c\u0e1d\u0e1e\u0e1f\u0e20\u0e21\u0e22\u0e23\u0e24\u0e25\u0e27\u0e28\u0e29\u0e2a\u0e2b\u0e2c\u0e2d\u0e2e\u0e2f\u0e30\u0e31\u0e32\u0e33\u0e34\u0e35\u0e36\u0e37\u0e38\u0e39\u0e40\u0e41\u0e42\u0e43\u0e44\u0e45\u0e46\u0e47\u0e48\u0e49\u0e4a\u0e4b\u0e4c\u0e4d\u2014\u2018\u2019\u201c\u201d",
        punctuations="!\"'(),-.:;?|~ ",
        phonemes="",
        is_unique=True,
        is_sorted=True,
    ),
    phoneme_cache_path=None,
    precompute_num_workers=12,
    start_by_longest=True,
    datasets=DATASETS_CONFIG_LIST,
    cudnn_benchmark=False,
    max_audio_len=SAMPLE_RATE * MAX_AUDIO_LEN_IN_SECONDS,
    mixed_precision=False,
    test_sentences=[
        [
            "ทดสอบการอ่านออกเสียงภาษาไทย",
            "VCTK_cv005",
            None,
            "th",
        ],
        [
            "ยักษ์ใหญ่ไล่ยักษ์เล็ก ยักษ์เล็กไล่ยักษ์ใหญ่",
            "VCTK_cv068",
            None,
            "th",
        ],
        [
            "ยายกินลำไย น้ำลายยายไหลย้อย",
            "VCTK_cv057",
            None,
            "th",
        ],
        [
            "ชามเขียวคว่ำเช้า ชามขาวคว่ำค่ำ",
            "VCTK_cv103",
            None,
            "th",
        ],
        [
            "หมอนลอยน้ำมา ว่ายน้ำไป ถอยหมอน",
            "VCTK_cv133",
            None,
            "th",
        ],
        [
            "เช้าฟาดผัดฟัก เย็นฟาดฟักผัด",
            "VCTK_cv128",
            None,
            "th",
        ],
    ],
    # # Enable the weighted sampler
    use_weighted_sampler=True,
    # # Ensures that all speakers are seen in the training batch equally no matter how many samples each speaker has
    weighted_sampler_attrs={"speaker_name": 1.0},
    # weighted_sampler_multipliers={},
    weighted_sampler_multipliers={"temp": None},

    # It defines the Speaker Consistency Loss (SCL) α to 9 like the paper
    speaker_encoder_loss_alpha=9.0,
)

# Training

In [9]:
# Load all the datasets samples and split traning and evaluation sets
train_samples, eval_samples = load_tts_samples(
    config.datasets,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

 | > Found 91809 files in /home/ming/Capstone/dubbing-ai/KhongKhunTTS/commonvoice-to-vctk
 | > Found 221131 files in /home/ming/Capstone/dubbing-ai/KhongKhunTTS/thai-central-to-vctk


In [10]:
# Init the model
model = Vits.init_from_config(config)

 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512


In [11]:
# Init the trainer and 🚀
trainer = Trainer(
    TrainerArgs(restore_path=RESTORE_PATH, skip_train_epoch=SKIP_TRAIN_EPOCH),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 16
 | > Num. of Torch Threads: 24
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/home/ming/Capstone/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-Experiment04-CV_and_TC-February-24-2025_12+39PM-f3ad324
 > Restoring from best_model_base.pth ...


 > `speakers.pth` is saved to /home/ming/Capstone/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-Experiment04-CV_and_TC-February-24-2025_12+39PM-f3ad324/speakers.pth.
 > `speakers_file` is updated in the config.json.
 > `language_ids.json` is saved to /home/ming/Capstone/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-Experiment04-CV_and_TC-February-24-2025_12+39PM-f3ad324/language_ids.json.
 > `language_ids_file` is updated in the config.json.


 > Restoring Model...
 > Restoring Optimizer...
 > Model restored from step 0
/home/ming/.cache/pypoetry/virtualenvs/khongkhuntts-Tu4HgoEE-py3.10/lib/python3.10/site-packages/trainer/trainer.py:561: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()

 > Model has 86831232 parameters


In [ ]:
trainer.fit()


 > EPOCH: 0/1000
 --> /home/ming/Capstone/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-Experiment04-CV_and_TC-February-24-2025_12+39PM-f3ad324




> DataLoader initialization
| > Tokenizer:
	| > add_blank: True
	| > use_eos_bos: False
	| > use_phonemes: False
| > Number of instances : 312684
 | > Preprocessing samples
 | > Max text length: 229
 | > Min text length: 1
 | > Avg text length: 41.13828446071553
 | 
 | > Max audio length: 160000.0
 | > Min audio length: 7775.5
 | > Avg audio length: 58788.89498500701
 | > Num. instances discarded samples: 31886
 | > Batch group size: 1536.
 > Using weighted sampler for attribute 'speaker_name' with alpha '1.0'
None



 > TRAINING (2025-02-24 12:39:44) 


 > Attribute weights for '['VCTK_cv001', 'VCTK_cv002', 'VCTK_cv003', 'VCTK_cv004', 'VCTK_cv005', 'VCTK_cv006', 'VCTK_cv007', 'VCTK_cv008', 'VCTK_cv009', 'VCTK_cv010', 'VCTK_cv011', 'VCTK_cv012', 'VCTK_cv013', 'VCTK_cv014', 'VCTK_cv015', 'VCTK_cv016', 'VCTK_cv018', 'VCTK_cv019', 'VCTK_cv020', 'VCTK_cv021', 'VCTK_cv022', 'VCTK_cv023', 'VCTK_cv024', 'VCTK_cv025', 'VCTK_cv027', 'VCTK_cv028', 'VCTK_cv029', 'VCTK_cv030', 'VCTK_cv031', 'VCTK_cv032', 'VCTK_cv033', 'VCTK_cv034', 'VCTK_cv035', 'VCTK_cv036', 'VCTK_cv037', 'VCTK_cv038', 'VCTK_cv040', 'VCTK_cv041', 'VCTK_cv042', 'VCTK_cv043', 'VCTK_cv044', 'VCTK_cv045', 'VCTK_cv046', 'VCTK_cv047', 'VCTK_cv050', 'VCTK_cv051', 'VCTK_cv053', 'VCTK_cv055', 'VCTK_cv056', 'VCTK_cv057', 'VCTK_cv058', 'VCTK_cv059', 'VCTK_cv060', 'VCTK_cv061', 'VCTK_cv062', 'VCTK_cv063', 'VCTK_cv064', 'VCTK_cv065', 'VCTK_cv066', 'VCTK_cv067', 'VCTK_cv068', 'VCTK_cv070', 'VCTK_cv071', 'VCTK_cv072', 'VCTK_cv073', 'VCTK_cv074', 'VCTK_cv075', 'VCTK_cv076', 'VCTK_cv077', 'VCTK_c

/home/ming/.cache/pypoetry/virtualenvs/khongkhuntts-Tu4HgoEE-py3.10/lib/python3.10/site-packages/torch/utils/data/sampler.py:77: UserWarning: `data_source` argument is not used and will be removed in 2.2.0.You may still have custom implementation that utilizes it.
  warnings.warn(
/home/ming/.cache/pypoetry/virtualenvs/khongkhuntts-Tu4HgoEE-py3.10/lib/python3.10/site-packages/torch/functional.py:709: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:873.)
  return _VF.stft(  # type: ignore[attr-defined]
/home/ming/.cache/pypoetry/virtualenvs/khongkhuntts-Tu4HgoEE-py3.10/lib/python3.10/site-packages/TTS/tts/models/vits.py:1273: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecat

...

/home/ming/.cache/pypoetry/virtualenvs/khongkhuntts-Tu4HgoEE-py3.10/lib/python3.10/site-packages/TTS/tts/models/vits.py:1455: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3725.)
  test_figures["{}-alignment".format(idx)] = plot_alignment(alignment.T, output_fig=False)

  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.06967727343241374 (+0)
     | > avg_loss_disc: 2.2079037030537925 (+0)
     | > avg_loss_disc_real_0: 0.11380600929260254 (+0)
     | > avg_loss_disc_real_1: 0.20239859819412231 (+0)
     | > avg_loss_disc_real_2: 0.19526173174381256 (+0)
     | > avg_loss_disc_real_3: 0.22384936610857645 (+0)
     | > avg_loss_disc_real_4: 0.1899478609363238 (+0)
     | > avg_

 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.07461659113566081 (+0.00493931770324707)
     | > avg_loss_disc: 2.177056829134623 (-0.030846873919169404)
     | > avg_loss_disc_real_0: 0.06320495717227459 (-0.05060105212032795)
     | > avg_loss_disc_real_1: 0.24001591900984445 (+0.03761732081572214)
     | > avg_loss_disc_real_2: 0.25857215871413547 (+0.06331042697032291)
     | > avg_loss_disc_real_3: 0.2099293445547422 (-0.013920021553834261)
     | > avg_loss_disc_real_4: 0.1902984231710434 (+0.00035056223471960357)
     | > avg_loss_disc_real_5: 0.24219710876544318 (+0.029105988641579955)
     | > avg_loss_0: 2.177056829134623 (-0.030846873919169404)
     | > avg_loss_spk_encoder: -5.812200864156087 (-0.27266414960225394)
     | > avg_loss_gen: 2.6606598695119223 (+0.15587302049001073)
     | > avg_loss_kl: 3.9792609214782715 (+0.223167578379313)
     | > avg_loss_feat: 5.900800546010335 (-0.20941662788391113)
     | > avg_loss_mel: 23.104993502298992 (-0.7933432261149065)
  

 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.2112428347269694 (+0.1366262435913086)
     | > avg_loss_disc: 2.1424872080485025 (-0.034569621086120605)
     | > avg_loss_disc_real_0: 0.1440671570599079 (+0.08086219988763332)
     | > avg_loss_disc_real_1: 0.15445065249999365 (-0.0855652665098508)
     | > avg_loss_disc_real_2: 0.21248953541119894 (-0.046082623302936526)
     | > avg_loss_disc_real_3: 0.1842486932873726 (-0.025680651267369597)
     | > avg_loss_disc_real_4: 0.20026318977276483 (+0.009964766601721436)
     | > avg_loss_disc_real_5: 0.2035919651389122 (-0.038605143626530974)
     | > avg_loss_0: 2.1424872080485025 (-0.034569621086120605)
     | > avg_loss_spk_encoder: -5.786310036977132 (+0.025890827178955078)
     | > avg_loss_gen: 2.710562070210775 (+0.04990220069885254)
     | > avg_loss_kl: 3.861521085103353 (-0.11773983637491847)
     | > avg_loss_feat: 6.232961734135945 (+0.33216118812561035)
     | > avg_loss_mel: 22.91112168629964 (-0.19387181599935133)
    

In [ ]:
trainer.save_checkpoint()